In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
cd /content/drive/MyDrive/spacetimeformer

/content/drive/MyDrive/spacetimeformer


In [ ]:
!pip install pip==24.0

In [ ]:
!sudo apt-get install build-essential
!sudo apt-get install libatlas-base-dev


In [ ]:
!pip install pystan --only-binary=:all:



In [ ]:
!pip install -r requirements.txt

In [ ]:
cd /content/drive/MyDrive/spacetimeformer

In [ ]:
!pip install -e .


In [ ]:
cd spacetimeformer

In [ ]:
# !python train.py spacetimeformer bus --start_token_len 3 --batch_size 5 \
# --gpus 0 --grad_clip_norm 1 --d_model 128 --d_ff 512 --enc_layers 5 \
# --dec_layers 4 --dropout_emb .3 --dropout_ff .3 --dropout_qkv 0 \
# --run_name spatiotemporal_metr-la --base_lr 1e-3 --l2_coeff 1e-2


In [ ]:
import sys
import os

sys.path.append('/content/drive/MyDrive/spacetimeformer')

In [ ]:
import spacetimeformer.spacetimeformer_model as stf
import torch

def create_model(config):
    # Define x_dim, yc_dim, yt_dim for the bus dataset
    if config.dset == "bus":
        x_dim = 6  # Input dimensions (features)
        yc_dim = 1  # Number of output features (you may adjust this based on your dataset)
        yt_dim = 1  # Number of targets (you may adjust this based on your dataset)
    else:
        raise ValueError("Unsupported dataset. Please use the 'bus' dataset.")

    # Set default values for model parameters if they are not defined in the config
    max_seq_len = getattr(config, 'max_seq_len', config.context_points + config.target_points if hasattr(config, 'context_points') and hasattr(config, 'target_points') else 24)
    start_token_len = getattr(config, 'start_token_len', 1)
    attn_factor = getattr(config, 'attn_factor', 1)
    d_model = getattr(config, 'd_model', 64)
    d_qk = getattr(config, 'd_qk', 64)
    d_v = getattr(config, 'd_v', 64)
    n_heads = getattr(config, 'n_heads', 8)
    enc_layers = getattr(config, 'enc_layers', 3)
    dec_layers = getattr(config, 'dec_layers', 3)
    d_ff = getattr(config, 'd_ff', 256)
    dropout_emb = getattr(config, 'dropout_emb', 0.1)
    dropout_attn_out = getattr(config, 'dropout_attn_out', 0.1)
    dropout_attn_matrix = getattr(config, 'dropout_attn_matrix', 0.1)
    dropout_qkv = getattr(config, 'dropout_qkv', 0.1)
    dropout_ff = getattr(config, 'dropout_ff', 0.1)
    pos_emb_type = getattr(config, 'pos_emb_type', 'abs')
    no_final_norm = getattr(config, 'no_final_norm', False)
    global_self_attn = getattr(config, 'global_self_attn', True)
    local_self_attn = getattr(config, 'local_self_attn', False)
    global_cross_attn = getattr(config, 'global_cross_attn', True)
    local_cross_attn = getattr(config, 'local_cross_attn', False)
    performer_kernel = getattr(config, 'performer_kernel', 'relu')
    performer_redraw_interval = getattr(config, 'performer_redraw_interval', 100)
    attn_time_windows = getattr(config, 'attn_time_windows', [1, 3, 6])
    use_shifted_time_windows = getattr(config, 'use_shifted_time_windows', False)
    norm = getattr(config, 'norm', 'layer')
    activation = getattr(config, 'activation', 'gelu')
    init_lr = getattr(config, 'init_lr', 1e-4)
    base_lr = getattr(config, 'base_lr', 1e-4)
    warmup_steps = getattr(config, 'warmup_steps', 4000)
    decay_factor = getattr(config, 'decay_factor', 0.1)
    initial_downsample_convs = getattr(config, 'initial_downsample_convs', 1)
    intermediate_downsample_convs = getattr(config, 'intermediate_downsample_convs', 1)
    embed_method = getattr(config, 'embed_method', 'spation-temporal')
    l2_coeff = getattr(config, 'l2_coeff', 1e-4)
    loss = getattr(config, 'loss', 'mse')
    class_loss_imp = getattr(config, 'class_loss_imp', 1.0)
    recon_loss_imp = getattr(config, 'recon_loss_imp', 1.0)
    time_emb_dim = getattr(config, 'time_emb_dim', 32)
    null_value = getattr(config, 'null_value', 0.0)
    pad_value = getattr(config, 'pad_value', 0.0)
    linear_window = getattr(config, 'linear_window', 6)
    use_revin = getattr(config, 'use_revin', False)
    linear_shared_weights = getattr(config, 'linear_shared_weights', False)
    use_seasonal_decomp = getattr(config, 'use_seasonal_decomp', False)
    no_val = getattr(config, 'no_val', False)
    no_time = getattr(config, 'no_time', False)
    no_space = getattr(config, 'no_space', False)
    no_given = getattr(config, 'no_given', False)
    recon_mask_skip_all = getattr(config, 'recon_mask_skip_all', False)
    recon_mask_max_seq_len = getattr(config, 'recon_mask_max_seq_len', 24)
    recon_mask_drop_seq = getattr(config, 'recon_mask_drop_seq', False)
    recon_mask_drop_standard = getattr(config, 'recon_mask_drop_standard', False)
    recon_mask_drop_full = getattr(config, 'recon_mask_drop_full', False)

    # Initialize the Spacetimeformer model
    forecaster = stf.spacetimeformer_model.Spacetimeformer_Forecaster(
        d_x=x_dim,
        d_yc=yc_dim,
        d_yt=yt_dim,
        max_seq_len=max_seq_len,
        start_token_len=start_token_len,
        attn_factor=attn_factor,
        d_model=d_model,
        d_queries_keys=d_qk,
        d_values=d_v,
        n_heads=n_heads,
        e_layers=enc_layers,
        d_layers=dec_layers,
        d_ff=d_ff,
        dropout_emb=dropout_emb,
        dropout_attn_out=dropout_attn_out,
        dropout_attn_matrix=dropout_attn_matrix,
        dropout_qkv=dropout_qkv,
        dropout_ff=dropout_ff,
        pos_emb_type=pos_emb_type,
        use_final_norm=not no_final_norm,
        global_self_attn=global_self_attn,
        local_self_attn=local_self_attn,
        global_cross_attn=global_cross_attn,
        local_cross_attn=local_cross_attn,
        performer_kernel=performer_kernel,
        performer_redraw_interval=performer_redraw_interval,
        attn_time_windows=attn_time_windows,
        use_shifted_time_windows=use_shifted_time_windows,
        norm=norm,
        activation=activation,
        init_lr=init_lr,
        base_lr=base_lr,
        warmup_steps=warmup_steps,
        decay_factor=decay_factor,
        initial_downsample_convs=initial_downsample_convs,
        intermediate_downsample_convs=intermediate_downsample_convs,
        embed_method=embed_method,
        l2_coeff=l2_coeff,
        loss=loss,
        class_loss_imp=class_loss_imp,
        recon_loss_imp=recon_loss_imp,
        time_emb_dim=time_emb_dim,
        null_value=null_value,
        pad_value=pad_value,
        linear_window=linear_window,
        use_revin=use_revin,
        linear_shared_weights=linear_shared_weights,
        use_seasonal_decomp=use_seasonal_decomp,
        use_val=not no_val,
        use_time=not no_time,
        use_space=not no_space,
        use_given=not no_given,
        recon_mask_skip_all=recon_mask_skip_all,
        recon_mask_max_seq_len=recon_mask_max_seq_len,
        recon_mask_drop_seq=recon_mask_drop_seq,
        recon_mask_drop_standard=recon_mask_drop_standard,
        recon_mask_drop_full=recon_mask_drop_full,
    )

    if torch.cuda.is_available():
        forecaster = forecaster.to('cuda')
        forecaster = torch.nn.SyncBatchNorm.convert_sync_batchnorm(forecaster)

    return forecaster


In [ ]:
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import torch

# Step 1: Create a DataFrame with the provided data
data = {
    "Datetime": ["6/30/2017 17:22"],
    "DirectionRef": [1],
    "PublishedLineName": [0.56248227],
    "NextStopPointName": [0.541666667],
    "TimeToArrival": [0.45],
}

# Convert to DataFrame
df = pd.DataFrame(data)

# Step 2: Preprocess the data
df['DirectionRef'] = df['DirectionRef'].astype(float)
df['PublishedLineName'] = df['PublishedLineName'].astype(float)
df['NextStopPointName'] = df['NextStopPointName'].astype(float)
df['TimeToArrival'] = df['TimeToArrival'].astype(float)

# Extract the relevant features as tensors
x_c = torch.tensor(df[['DirectionRef', 'PublishedLineName', 'NextStopPointName']].values, dtype=torch.float32)
y_c = torch.tensor(df['TimeToArrival'].values, dtype=torch.float32)
x_t = x_c.clone()

# Step 3: Reshape tensors to match model input shape
batch_size = 1
sequence_length = 12
feature_dim_x = x_c.shape[1] if x_c.dim() > 1 else 1
feature_dim_y = 1

x_c = x_c.unsqueeze(0).repeat(batch_size, sequence_length, 1)
y_c = y_c.unsqueeze(0).unsqueeze(-1).repeat(batch_size, sequence_length, feature_dim_y)
x_t = x_t.unsqueeze(0).repeat(batch_size, sequence_length, 1)

# Initialize a MinMaxScaler for scaling
scaler = MinMaxScaler()
scaler.fit(df[['TimeToArrival']])

# Function to apply scaling to the data
def scale_data(y):
    return scaler.transform(y.reshape(-1, 1)).flatten()

# Check the shapes of the tensors
print("x_c shape:", x_c.shape)
print("y_c shape:", y_c.shape)
print("x_t shape:", x_t.shape)

# Step 4: Load the trained model from the checkpoint
ckpt_path = './data/STF_LOG_DIR/spatiotemporal_metr-la_31b54772/spatiotemporal_metr-laepoch=00.ckpt'

class Args:
    def __init__(self):
        self.dset = "bus"
        self.model = "spacetimeformer"
        self.null_value = 0.0
        self.pad_value = 0.0
        self.gpus = 1
        self.grad_clip_norm = 1.0
        self.debug = False
        self.accumulate = 1
        self.limit_val_batches = 1
        self.wandb = False
        self.context_points = 12
        self.target_points = 5
        self.start_token_len = 1
        self.attn_factor = 1
        self.d_model = 64
        self.d_qk = 64
        self.d_v = 64
        self.n_heads = 8
        self.enc_layers = 3
        self.dec_layers = 3
        self.d_ff = 256
        self.dropout_emb = 0.1
        self.dropout_attn_out = 0.1
        self.dropout_attn_matrix = 0.1
        self.dropout_qkv = 0.1
        self.dropout_ff = 0.1
        self.global_self_attn = "none"
        self.local_self_attn = "none"
        self.global_cross_attn = "none"
        self.local_cross_attn = "none"
        self.performer_kernel = None
        self.performer_redraw_interval = None
        self.init_lr = 0.001
        self.base_lr = 0.0001
        self.warmup_steps = 500
        self.decay_factor = 0.1
        self.initial_downsample_convs = 0
        self.intermediate_downsample_convs = 0
        self.embed_method = "spatio-temporal"
        self.l2_coeff = 1e-5
        self.loss = "mse"
        self.class_loss_imp = 1.0
        self.recon_loss_imp = 1.0
        self.time_emb_dim = 16
        self.linear_window = None
        self.use_revin = False
        self.linear_shared_weights = False
        self.use_seasonal_decomp = False
        self.no_val = False
        self.no_time = False
        self.no_space = False
        self.no_given = False
        self.recon_mask_skip_all = False
        self.recon_mask_max_seq_len = None
        self.recon_mask_drop_seq = False
        self.recon_mask_drop_standard = False
        self.recon_mask_drop_full = False

# Instantiate the Args class
args = Args()

# Step 5: Create the model (assuming create_model exists)
forecaster = create_model(args)

# Step 6: Load the model from the checkpoint
forecaster = forecaster.load_from_checkpoint(ckpt_path, args=args)

# Step 7: Set the scaler in the model
forecaster.set_scaler(scale_data)

# Step 8: Move the model and tensors to the appropriate device
device = torch.device('cpu')  # Change to 'cuda' if using GPU
forecaster.to(device)
x_c = x_c.to(device)
y_c = y_c.to(device)
x_t = x_t.to(device)

# Step 9: Make predictions
yt_pred = forecaster.predict(x_c=x_c, y_c=y_c, x_t=x_t)

# Step 10: Output the predictions
print("Predicted Time to Arrival:", yt_pred)
